# Explorative Datenanalyse (EDA)

In [ ]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.feature_selection import f_classif
from sklearn.ensemble import IsolationForest


from notebooks.edahelper import fill_spectral_nans

from math import ceil

# Loading Data

In [ ]:
#place test and training data in the data/raw folder
df_train = pd.read_csv(Path.cwd() / "data" / "raw" / "train.csv", index_col=0)
df_test = pd.read_csv(Path.cwd() / "data" / "raw" / "test.csv", index_col=0)

In [ ]:
df_train.head()

# NaN values
## Empty bands
-> dropping completely 

In [ ]:
empty_cols = df_train.columns[df_train.isna().all()] #.tolist()
df_train = df_train.drop(columns=empty_cols)
df_test = df_test.drop(columns=empty_cols)

print("Dropped columns:\n", empty_cols)


## Single Missing values

In [ ]:
df_train[df_train.isnull().any(axis=1)][df_train.columns[df_train.isna().any()]]

In [ ]:
df_test[df_test.isnull().any(axis=1)][df_test.columns[df_test.isna().any()]]

In [ ]:
nan_columns = df_train.columns[df_train.isna().any()]
print(nan_columns)

In [ ]:
# [link to line diagram] ->  Filling with values from neighboring columns better then with mean of the column (only as backup)
df_train = fill_spectral_nans(df_train, 15, 45)
df_test = fill_spectral_nans(df_test, 15, 45)

## Duplicate

In [ ]:
#Absolute Duplikate
df_train[df_train.drop('id', axis=1).duplicated(keep=False)]

In [ ]:
# Drop duplicates based on all columns except 'id'
df_train.drop_duplicates(subset=df_train.columns.difference(['id']), inplace=True)

In [ ]:

df_train[df_train['X2315'].duplicated(False)]

# Vergleich Test und Trainingsdaten

In [ ]:
df_compare = pd.concat(
    [df_train.assign(type="train"), df_test.assign(type="test")],
    ignore_index=True,
)

In [ ]:
comparison = (
    df_compare.melt(
        id_vars="type",
        value_vars=["AEZ", "Month"],
        var_name="feature",
        value_name="category",
    )
    .groupby(["feature", "category", "type"], dropna=False)
    .size()
    .reset_index(name="count")
)

fig = px.bar(
    comparison,
    x="category",
    y="count",
    color="type",
    barmode="group",
    facet_col="feature",
    labels={"category": "Kategorie", "count": "Anzahl der Beobachtungen", "type": "Datensatz"},
    title="Verteilung von AEZ und Month: Training und Test (absolute Häufigkeiten)"
)
fig.show()

In [ ]:
comparison = (
    df_compare.melt(
        id_vars="type",
        value_vars=["AEZ", "Month"],
        var_name="feature",
        value_name="category",
    )
    .groupby(["feature", "category", "type"], dropna=False)
    .size()
    .reset_index(name="count")
)

comparison["relative_share"] = comparison["count"] / comparison["type"].map(
    df_compare["type"].value_counts()
) * 100

fig = px.bar(
    comparison,
    x="category",
    y="relative_share",
    color="type",
    barmode="group",
    facet_col="feature",
    labels={
        "category": "Kategorie",
        "relative_share": "Anteil am Datensatz (%)",
        "type": "Datensatz",
    },
    title="Verteilung von AEZ und Month: Training und Test (relative Anteile)"
)
fig.show()

In [ ]:


bands = sorted(
    (
        column
        for column in set(df_train.columns) & set(df_test.columns)
        if column.startswith("X") and column[1:].isdigit()
    ),
    key=lambda column: int(column[1:]),
)

if not bands:
    raise ValueError("No shared spectral bands found between train and test.")

wavelengths_nm = [int(band[1:]) for band in bands]

fig = go.Figure()

for label, df, color in [
    ("Training", df_train, "blue"),
    ("Test", df_test, "orange"),
]:
    fig.add_trace(
        go.Box(
            x=wavelengths_nm * len(df),
            y=df[bands].to_numpy().ravel(),
            name=label,
            offsetgroup=label,
            marker_color=color,
            boxpoints=False,
            hovertemplate=(
                f"{label}<br>Wellenlänge=%{{x}} nm"
                "<br>Reflektanz=%{y}<extra></extra>"
            ),
        )
    )

fig.update_layout(
    title="Reflektanzverteilung nach Wellenlänge: Training und Test",
    xaxis={"title": "Wellenlänge [nm]", "type": "linear"},
    yaxis_title="Reflektanz [%]",
    boxmode="group",
)
fig.show()

In [ ]:
import plotly.graph_objects as go

bands = sorted(
    (
        column
        for column in set(df_train.columns) & set(df_test.columns)
        if column.startswith("X") and column[1:].isdigit()
    ),
    key=lambda column: int(column[1:]),
)

if not bands:
    raise ValueError("No shared spectral bands found between train and test.")

wavelengths_nm = [int(band[1:]) for band in bands]
train_mean = df_train[bands].mean()
test_mean = df_test[bands].mean()

fig = go.Figure()

for label, mean_spectrum, color in [
    ("Mittelwert Training", train_mean, "blue"),
    ("Mittelwert Test", test_mean, "orange"),
]:
    x_values = []
    y_values = []

    for index, (wavelength, band) in enumerate(zip(wavelengths_nm, bands)):
        if index > 0 and wavelength - wavelengths_nm[index - 1] > 50:
            x_values.append(None)
            y_values.append(None)

        x_values.append(wavelength)
        y_values.append(mean_spectrum[band])

    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=y_values,
            mode="lines",
            name=label,
            line={"color": color},
            hovertemplate="Wellenlänge=%{x} nm<br>Mittlere Reflektanz=%{y}<extra></extra>",
        )
    )

fig.update_layout(
    title="Mittleres Spektrum: Training und Test",
    xaxis_title="Wellenlänge [nm]",
    yaxis_title="Mittlere Reflektanz [%]",
)
fig.show()

In [ ]:


bands = sorted(
    (
        column
        for column in set(df_train.columns) & set(df_test.columns)
        if column.startswith("X") and column[1:].isdigit()
    ),
    key=lambda column: int(column[1:]),
)

if not bands:
    raise ValueError("No shared spectral bands found between train and test.")

wavelengths_nm = [int(band[1:]) for band in bands]

fig = go.Figure()

for label, df, color in [
    ("Training", df_train, "blue"),
    ("Test", df_test, "orange"),
]:
    fig.add_trace(
        go.Box(
            x=wavelengths_nm * len(df),
            y=df[bands].to_numpy().ravel(),
            name=label,
            offsetgroup=label,
            marker_color=color,
            boxpoints=False,
            hovertemplate=(
                f"{label}<br>Wellenlänge=%{{x}} nm"
                "<br>Reflektanz=%{y}<extra></extra>"
            ),
        )
    )

fig.update_layout(
    title="Reflektanzverteilung nach Wellenlänge: Training und Test",
    xaxis={"title": "Wellenlänge [nm]", "type": "linear"},
    yaxis_title="Reflektanz [%]",
    boxmode="group",
)
fig.show()

# Similarities in test dataset

In [ ]:
bands = sorted((column for column in df_train.columns if column.startswith("X") and column[1:].isdigit()),
    key=lambda column: int(column[1:]),
    )

spectra = df_train[bands].to_numpy(dtype=float)

if not np.isfinite(spectra).all():
    raise ValueError("Handle missing or infinite frequency values first.")

sample_ids = (
    df_train["id"].to_numpy()
    if "id" in df_train.columns
    else df_train.index.to_numpy()
)
crops = df_train["Crop"].to_numpy()
stages = df_train["Stage"].to_numpy()

pair_i, pair_j = np.triu_indices(len(df_train), k=1)


def make_pair_table(values, metric_name):
    """Format the upper triangle of a pairwise metric matrix as sample pairs."""
    scores = values[pair_i, pair_j]
    valid = np.isfinite(scores)

    i = pair_i[valid]
    j = pair_j[valid]

    return pd.DataFrame(
        {
            "id x": sample_ids[i],
            "id y": sample_ids[j],
            metric_name: scores[valid],
            "crop x": crops[i],
            "crop y": crops[j],
            "stage x": stages[i],
            "stage y": stages[j],
        }
    )

## Pearson correlation — higher means more similar:

In [ ]:
centered = spectra - spectra.mean(axis=1, keepdims=True)
norms = np.linalg.norm(centered, axis=1)

if np.any(norms == 0):
    raise ValueError("Pearson correlation is undefined for a constant spectrum.")

normalized = centered / norms[:, None]
pearson_matrix = normalized @ normalized.T
pearson_matrix = np.clip(pearson_matrix, -1.0, 1.0)

pearson_pairs = make_pair_table(pearson_matrix, "pearson correlation")
display(pearson_pairs)

## Spectral Angle Mapper (SAM) — angle in degrees; lower means more similar:



In [ ]:
norms = np.linalg.norm(spectra, axis=1)

if np.any(norms == 0):
    raise ValueError("SAM is undefined for a zero spectrum.")

normalized = spectra / norms[:, None]
cosine_matrix = np.clip(normalized @ normalized.T, -1.0, 1.0)
sam_matrix = np.degrees(np.arccos(cosine_matrix))

sam_pairs = make_pair_table(sam_matrix, "SAM angle (degrees)")
display(sam_pairs)

## Euclidean distance — lower means more similar; sensitive to reflectance scale:

In [ ]:
squared_norms = np.sum(spectra**2, axis=1)
squared_distances = (
    squared_norms[:, None] + squared_norms[None, :] - 2 * (spectra @ spectra.T)
)
euclidean_matrix = np.sqrt(np.maximum(squared_distances, 0.0))

euclidean_pairs = make_pair_table(euclidean_matrix, "euclidean distance")
display(euclidean_pairs)

## Spectral Information Divergence (SID) — lower means more similar; requires nonnegative spectra:

In [ ]:
if np.any(spectra < 0):
    raise ValueError("SID requires nonnegative spectral values.")

# Convert each spectrum to a probability distribution across its bands.
positive = np.maximum(spectra, 1e-12)
probabilities = positive / positive.sum(axis=1, keepdims=True)
log_probabilities = np.log(probabilities)

cross_terms = probabilities @ log_probabilities.T
self_terms = np.sum(probabilities * log_probabilities, axis=1)
sid_matrix = (
    self_terms[:, None]
    + self_terms[None, :]
    - cross_terms
    - cross_terms.T
)
sid_matrix = np.maximum(sid_matrix, 0.0)

sid_pairs = make_pair_table(sid_matrix, "SID")
display(sid_pairs)

## Cosine similarity 

In [ ]:
norms = np.linalg.norm(spectra, axis=1)

if np.any(norms == 0):
    raise ValueError("Cosine similarity is undefined for a zero spectrum.")

normalized = spectra / norms[:, None]
cosine_matrix = np.clip(normalized @ normalized.T, -1.0, 1.0)

cosine_pairs = make_pair_table(cosine_matrix, "cosine similarity")
display(cosine_pairs)

In [ ]:
cosine_pairs[(cosine_pairs["cosine similarity"] > 0.99) & (cosine_pairs["crop x"] != cosine_pairs["crop y"]) & (cosine_pairs["stage x"] == cosine_pairs["stage y"])].sort_values(by="cosine similarity", ascending=False)

In [ ]:
cosine_pairs[(cosine_pairs["cosine similarity"] > 0.99) & (cosine_pairs["stage x"] != cosine_pairs["stage y"]) & (cosine_pairs["crop x"] == cosine_pairs["crop y"])].sort_values(by="cosine similarity", ascending=False)

In [ ]:
cosine_pairs[(cosine_pairs["cosine similarity"] > 0.99) & ((cosine_pairs["stage x"] != cosine_pairs["stage y"]) | (cosine_pairs["crop x"] != cosine_pairs["crop y"]))].sort_values(by="cosine similarity", ascending=False)

# Klassenverteilungen und Statifizierung der Daten
## Verteilungen Crop, Stage

In [ ]:
crop_stage_counts = pd.crosstab(df_train["Crop"], df_train["Stage"])

fig = px.imshow(
    crop_stage_counts,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    labels={"x": "Stage", "y": "Crop", "color": "Anzahl"},
    title="Anzahl der Trainingsproben je Crop und Stage",
)
fig.show()

In [ ]:
# AEZ × Crop
aez_crop = pd.crosstab(df_train["AEZ"], df_train["Crop"]).sort_index()

fig = px.imshow(
    aez_crop,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    labels={"x": "Crop", "y": "AEZ", "color": "Anzahl"},
    title="Trainingsproben nach AEZ und Crop",
)
fig.show()

# Month × Stage, one panel per crop
crops = sorted(df_train["Crop"].dropna().unique())
months = sorted(df_train["Month"].dropna().unique())
stages = sorted(df_train["Stage"].dropna().unique())

n_cols = 3
n_rows = ceil(len(crops) / n_cols)
fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=crops,
    shared_xaxes=True,
    shared_yaxes=True,
)

for i, crop in enumerate(crops):
    row, col = divmod(i, n_cols)
    counts = pd.crosstab(
        df_train.loc[df_train["Crop"] == crop, "Month"],
        df_train.loc[df_train["Crop"] == crop, "Stage"],
    ).reindex(index=months, columns=stages, fill_value=0)

    fig.add_trace(
        go.Heatmap(
            z=counts.values,
            x=stages,
            y=months,
            text=counts.values,
            texttemplate="%{text}",
            coloraxis="coloraxis",
        ),
        row=row + 1,
        col=col + 1,
    )

fig.update_layout(
    title="Trainingsproben nach Month und Stage, aufgeteilt nach Crop",
    coloraxis={"colorscale": "Blues"},
)
fig.update_xaxes(
    title_text="Stage",
    tickmode="array",
    tickvals=stages,
    ticktext=stages,
    tickangle=45,
    row=n_rows,
)
fig.update_yaxes(
    title_text="Month",
    tickmode="array",
    tickvals=months,
    ticktext=[str(month) for month in months],
)
fig.show()

## -> Stratifizierung der Trainings- und Testdaten nach Crop und Stage

In [ ]:
crop_stage_counts = (
    df_train.groupby(["Crop", "Stage"], observed=True)
    .size()
    .rename("absolute_count")
    .to_frame()
)
crop_stage_counts["relative_count"] = (
    crop_stage_counts["absolute_count"] / len(df_train)
)

crop_stage_counts.round(2)

In [ ]:

crop_stage = pd.MultiIndex.from_frame(df_train[["Crop", "Stage"]])
strata = pd.factorize(crop_stage)[0]

df_train_split, df_validation = train_test_split(
    df_train,
    test_size=0.2,
    stratify=strata,
)

In [ ]:
crop_stage_counts = (
    df_validation.groupby(["Crop", "Stage"], observed=True)
    .size()
    .rename("absolute_count")
    .to_frame()
)
crop_stage_counts["relative_count"] = (
    crop_stage_counts["absolute_count"] / len(df_validation)
)

crop_stage_counts.round(2)

# Bänder
Im Datensatz
437 - 923 <br>
983 - 1094 br>
1175 - 1316 <br>
1518 - 1760 <br>
2063 - 2345 <br>


Google:
400 – 450 nm | Coastal Aerosol / Deep Blue <br>
450 – 510 nm | Blue <br>
500 – 600 nm | Green<br>
600 – 700 nm | Red<br>
700 – 800 nm | Red Edge<br>
800 – 1,000 nm | Near-Infrared (NIR)<br>
1,300 – 1,400 nm | Cirrus 
1,550 – 1,750 nm | SWIR 1<br>
2,100 – 2,300 nm | SWIR 2<br>





## 16.1 Korrelationsmatrix der Bänder

In [ ]:
bands = sorted(
    (
        column
        for column in df_train.columns
        if column.startswith("X") and column[1:].isdigit()
    ),
    key=lambda column: int(column[1:]),
)
wavelengths_nm = [int(band[1:]) for band in bands]

spectra = df_train[bands].astype(float)
if not np.isfinite(spectra.to_numpy()).all():
    raise ValueError("Handle missing or infinite band values before calculating correlations.")

band_corr = spectra.corr(method="pearson")



tick_indices = np.arange(0, len(wavelengths_nm), 10)

fig = go.Figure(
    go.Heatmap(
        z=band_corr.to_numpy(),
        x=wavelengths_nm,
        y=wavelengths_nm,
        zmin=-1,
        zmax=1,
        zmid=0,
        colorscale="RdBu",
        reversescale=True,
        colorbar={"title": "Pearson-r"},
        hovertemplate=(
            "Wellenlänge x: %{x} nm<br>"
            "Wellenlänge y: %{y} nm<br>"
            "Korrelation: %{z:.3f}<extra></extra>"
        ),
    )
)

fig.update_layout(
    title="Pearson-Korrelation zwischen Spektralbändern",
    xaxis={
        "title": "Wellenlänge [nm]",
        "tickmode": "array",
        "tickvals": [wavelengths_nm[i] for i in tick_indices],
    },
    yaxis={
        "title": "Wellenlänge [nm]",
        "tickmode": "array",
        "tickvals": [wavelengths_nm[i] for i in tick_indices],
    },
    width=900,
    height=800,
)
fig.show()

In [ ]:
# Correlation with neighbouring bands

df_band_corr_neighbour = pd.DataFrame(
    {
        "band": bands,
        "wavelength_nm": wavelengths_nm,
        "left_band": [bands[i - 1] if i > 0 else None for i in range(len(bands))],
        "corr_left": [band_corr.iloc[i, i - 1] if i > 0 else np.nan for i in range(len(bands))],
        "right_band": [bands[i + 1] if i + 1 < len(bands) else None for i in range(len(bands))],
        "corr_right": [
            band_corr.iloc[i, i + 1] if i + 1 < len(bands) else np.nan
            for i in range(len(bands))
        ],
    }
)
df_band_corr_neighbour["mean_neighbour_corr"] = df_band_corr_neighbour[
    ["corr_left", "corr_right"]
].mean(axis=1)
df_band_corr_neighbour["mean_abs_neighbour_corr"] = df_band_corr_neighbour[
    ["corr_left", "corr_right"]
].abs().mean(axis=1)

df_band_corr_neighbour.round(4)
df_band_corr_neighbour.sort_values(by="mean_abs_neighbour_corr", ascending=False)

## 16.2 PCA: erklärte Varianz und 2D-Projektion nach Crop/Stage (optional UMAP/t-SNE)

In [ ]:
from sklearn.decomposition import PCA

bands = sorted(
    (
        column
        for column in df_train.columns
        if column.startswith("X") and column[1:].isdigit()
    ),
    key=lambda column: int(column[1:]),
)

X = df_train[bands].to_numpy(dtype=float)
if not np.isfinite(X).all():
    raise ValueError("Handle missing or infinite band values before PCA.")

# Reflectance bands share units, so retain their original scale and center them with PCA.
pca = PCA()
X_pca = pca.fit_transform(X)

variance_pct = pca.explained_variance_ratio_ * 100
components = np.arange(1, len(variance_pct) + 1)

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=components,
        y=variance_pct,
        name="Varianz je PC",
    )
)
fig.add_trace(
    go.Scatter(
        x=components,
        y=np.cumsum(variance_pct),
        name="Kumulierte Varianz",
        yaxis="y2",
    )
)
fig.update_layout(
    title="PCA: erklärte Varianz",
    xaxis_title="Hauptkomponente",
    yaxis_title="Erklärte Varianz je PC [%]",
    yaxis2={
        "title": "Kumulierte Varianz [%]",
        "overlaying": "y",
        "side": "right",
        "range": [0, 100],
    },
)
fig.show()

pca_plot = df_train[["Crop", "Stage"]].copy()
pca_plot["PC1"] = X_pca[:, 0]
pca_plot["PC2"] = X_pca[:, 1]
pc1_pct, pc2_pct = variance_pct[:2]

fig_crop = px.scatter(
    pca_plot,
    x="PC1",
    y="PC2",
    color="Crop",
    hover_data=["Stage"],
    title="PCA-Projektion nach Crop",
    labels={
        "PC1": f"PC1 ({pc1_pct:.1f} % Varianz)",
        "PC2": f"PC2 ({pc2_pct:.1f} % Varianz)",
    },
)
fig_crop.show()

fig_stage = px.scatter(
    pca_plot,
    x="PC1",
    y="PC2",
    color="Stage",
    hover_data=["Crop"],
    title="PCA-Projektion nach Stage",
    labels={
        "PC1": f"PC1 ({pc1_pct:.1f} % Varianz)",
        "PC2": f"PC2 ({pc2_pct:.1f} % Varianz)",
    },
)
fig_stage.show()

## 16.3 Erste Hinweise für Bandreduktion (M3)
PCA mit 19 PC liefert eine erklärte Varianz von >99,9%. 

# 17

## 17.1 Mittlere Spektren ± Std je Kultur, je Stadium innerhalb jeder Kultur, je AEZ


In [ ]:
bands = sorted(
    (
        column
        for column in df_train.columns
        if column.startswith("X") and column[1:].isdigit()
    ),
    key=lambda column: int(column[1:]),
)
wavelengths_nm = [int(band[1:]) for band in bands]

spectra = df_train[bands].astype(float)
if not np.isfinite(spectra.to_numpy()).all():
    raise ValueError("Handle missing or infinite band values before plotting spectra.")

spectral_segments = []
segment_start = 0
for index in range(1, len(wavelengths_nm)):
    if wavelengths_nm[index] - wavelengths_nm[index - 1] > 50:
        spectral_segments.append(slice(segment_start, index))
        segment_start = index
spectral_segments.append(slice(segment_start, len(wavelengths_nm)))


def add_mean_std(fig, group, label, color, row=None, col=None, showlegend=True):
    mean = group[bands].mean().to_numpy()
    std = group[bands].std().to_numpy()
    count = len(group)
    rgb = tuple(int(color[index : index + 2], 16) for index in (1, 3, 5))
    trace_args = {} if row is None else {"row": row, "col": col}

    for segment_index, segment in enumerate(spectral_segments):
        wavelengths = wavelengths_nm[segment]
        segment_mean = mean[segment]
        segment_std = std[segment]

        fig.add_trace(
            go.Scatter(
                x=wavelengths,
                y=segment_mean - segment_std,
                mode="lines",
                line={"width": 0},
                showlegend=False,
                hoverinfo="skip",
            ),
            **trace_args,
        )
        fig.add_trace(
            go.Scatter(
                x=wavelengths,
                y=segment_mean + segment_std,
                mode="lines",
                line={"width": 0},
                fill="tonexty",
                fillcolor=f"rgba({rgb[0]}, {rgb[1]}, {rgb[2]}, 0.18)",
                showlegend=False,
                hoverinfo="skip",
            ),
            **trace_args,
        )
        fig.add_trace(
            go.Scatter(
                x=wavelengths,
                y=segment_mean,
                mode="lines",
                name=label,
                line={"color": color},
                showlegend=showlegend and segment_index == 0,
                legendgroup=label,
                customdata=np.column_stack(
                    [segment_std, np.full(len(wavelengths), count)]
                ),
                hovertemplate=(
                    "Wellenlänge=%{x} nm<br>Mittelwert=%{y:.2f}"
                    "<br>Standardabweichung=%{customdata[0]:.2f}<br>n=%{customdata[1]}"
                    "<extra>%{fullData.name}</extra>"
                ),
            ),
            **trace_args,
        )

# Mean spectra by crop.
crop_groups = list(df_train.groupby("Crop", sort=True, observed=True))
fig_crop = go.Figure()
for index, (crop, group) in enumerate(crop_groups):
    add_mean_std(
        fig_crop,
        group,
        f"{crop} (n={len(group)})",
        px.colors.qualitative.Plotly[index % len(px.colors.qualitative.Plotly)],
    )
fig_crop.update_layout(
    title="Mittlere Spektralsignatur ± 1 Standardabweichung nach Crop",
    xaxis_title="Wellenlänge [nm]",
    yaxis_title="Reflektanz",
    width=1000,
    height=550,
)
fig_crop.update_xaxes(tickmode="array", tickvals=wavelengths_nm[::10])
fig_crop.show()

# Mean spectra by stage, pooled across crops.
stage_groups = list(df_train.groupby("Stage", sort=True, observed=True))
fig_stage_mean = go.Figure()
for index, (stage, group) in enumerate(stage_groups):
    add_mean_std(
        fig_stage_mean,
        group,
        f"{stage} (n={len(group)})",
        px.colors.qualitative.Plotly[index % len(px.colors.qualitative.Plotly)],
    )
fig_stage_mean.update_layout(
    title="Mittlere Spektralsignatur ± 1 Standardabweichung nach Stage (über Crop zusammengefasst)",
    xaxis_title="Wellenlänge [nm]",
    yaxis_title="Reflektanz",
    width=1000,
    height=550,
)
fig_stage_mean.update_xaxes(tickmode="array", tickvals=wavelengths_nm[::10])
fig_stage_mean.show()

# Mean spectra by stage within each crop.
crops = [crop for crop, _ in crop_groups]
stages = sorted(df_train["Stage"].dropna().unique())
stage_colors = {
    stage: px.colors.qualitative.Plotly[index % len(px.colors.qualitative.Plotly)]
    for index, stage in enumerate(stages)
}
n_cols = 2
n_rows = ceil(len(crops) / n_cols)
fig_stage = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[
        f"{crop} (n={len(group)})" for crop, group in crop_groups
    ],
    shared_xaxes=True,
    shared_yaxes=True,
)
shown_stages = set()
for index, crop in enumerate(crops):
    row, col = divmod(index, n_cols)
    crop_data = df_train.loc[df_train["Crop"] == crop]
    for stage, group in crop_data.groupby("Stage", sort=True, observed=True):
        add_mean_std(
            fig_stage,
            group,
            stage,
            stage_colors[stage],
            row=row + 1,
            col=col + 1,
            showlegend=stage not in shown_stages,
        )
        shown_stages.add(stage)
fig_stage.update_layout(
    title="Mittlere Spektralsignatur ± 1 Standardabweichung nach Stage innerhalb jedes Crop",
    yaxis_title="Reflektanz",
    height=350 * n_rows,
    width=1000,
)
fig_stage.update_xaxes(
    title_text="Wellenlänge [nm]",
    tickmode="array",
    tickvals=wavelengths_nm[::10],
)
fig_stage.update_yaxes(title_text="Reflektanz")
fig_stage.show()

# AEZ signatures are pooled over crop and stage; differences may reflect group composition.
aez_groups = list(df_train.groupby("AEZ", sort=True, observed=True))
fig_aez = go.Figure()
for index, (aez, group) in enumerate(aez_groups):
    add_mean_std(
        fig_aez,
        group,
        f"AEZ {aez} (n={len(group)})",
        px.colors.qualitative.Plotly[index % len(px.colors.qualitative.Plotly)],
    )
fig_aez.update_layout(
    title="Mittlere Spektralsignatur ± 1 Standardabweichung nach AEZ (über Crop und Stage zusammengefasst)",
    xaxis_title="Wellenlänge [nm]",
    yaxis_title="Reflektanz",
    width=1000,
    height=550,
)
fig_aez.update_xaxes(tickmode="array", tickvals=wavelengths_nm[::10])
fig_aez.show()

## 17.2

In [ ]:

bands = sorted(
    (
        column
        for column in df_train.columns
        if column.startswith("X") and column[1:].isdigit()
    ),
    key=lambda column: int(column[1:]),
)
wavelengths_nm = [int(band[1:]) for band in bands]
spectra = df_train[bands].astype(float)
if not np.isfinite(spectra.to_numpy()).all():
    raise ValueError("Handle missing or infinite band values before calculating ANOVA F-scores.")


def anova_f_by_band(frame, target):
    """Calculate one-way ANOVA F-statistics independently for each spectral band."""
    if frame[target].nunique() < 2:
        raise ValueError(f"At least two observed classes are required for {target}.")
    scores, _ = f_classif(frame[bands], frame[target])
    if not np.isfinite(scores).all():
        raise ValueError(f"ANOVA returned non-finite F-statistics for {target}.")
    return scores


def add_wavelength_gaps(values):
    plotted_values = []
    for index, value in enumerate(values):
        if index and wavelengths_nm[index] - wavelengths_nm[index - 1] > 50:
            plotted_values.append(None)
        plotted_values.append(value)
    return plotted_values


crop_f_scores = anova_f_by_band(df_train, "Crop")
df_band_anova_crop = pd.DataFrame(
    {"wavelength_nm": wavelengths_nm, "F_statistic": crop_f_scores}
)

fig_crop_f = go.Figure(
    go.Scatter(
        x=add_wavelength_gaps(wavelengths_nm),
        y=add_wavelength_gaps(crop_f_scores),
        mode="lines",
        name="Crop",
        customdata=add_wavelength_gaps(bands),
        hovertemplate=(
            "Band=%{customdata}<br>Wellenlänge=%{x} nm"
            "<br>ANOVA F=%{y:.2f}<extra></extra>"
        ),
    )
)
fig_crop_f.update_layout(
    title="Univariate ANOVA-F-Statistik je Wellenlänge für Crop",
    xaxis_title="Wellenlänge [nm]",
    yaxis_title="ANOVA-F-Statistik (höher bedeutet stärkere univariate Trennung)",
    width=1000,
    height=500,
)
fig_crop_f.update_xaxes(tickmode="array", tickvals=wavelengths_nm[::10])
fig_crop_f.show()

crops = sorted(df_train["Crop"].dropna().unique())
stages = sorted(df_train["Stage"].dropna().unique())
stage_colors = {
    stage: px.colors.qualitative.Plotly[index % len(px.colors.qualitative.Plotly)]
    for index, stage in enumerate(stages)
}

stage_f_scores = anova_f_by_band(df_train, "Stage")
df_band_anova_stage_all = pd.DataFrame(
    {"wavelength_nm": wavelengths_nm, "F_statistic": stage_f_scores}
)
fig_stage_univariate = go.Figure(
    go.Scatter(
        x=add_wavelength_gaps(wavelengths_nm),
        y=add_wavelength_gaps(stage_f_scores),
        mode="lines",
        name="Stage",
        customdata=add_wavelength_gaps(bands),
        hovertemplate=(
            "Band=%{customdata}<br>Wellenlänge=%{x} nm"
            "<br>ANOVA F=%{y:.2f}<extra></extra>"
        ),
    )
)
fig_stage_univariate.update_layout(
    title="Univariate ANOVA-F-Statistik je Wellenlänge für Stage",
    xaxis_title="Wellenlänge [nm]",
    yaxis_title="ANOVA-F-Statistik (höher bedeutet stärkere Trennung der Stadien)",
    width=1000,
    height=500,
)
fig_stage_univariate.update_xaxes(
    tickmode="array",
    tickvals=wavelengths_nm[::10],
)
fig_stage_univariate.update_yaxes(type="log")
fig_stage_univariate.show()
stage_by_crop_rows = []
fig_stage_by_crop = go.Figure()
crop_colors = {
    crop: px.colors.qualitative.Plotly[index % len(px.colors.qualitative.Plotly)]
    for index, crop in enumerate(crops)
}
for crop in crops:
    crop_data = df_train.loc[df_train["Crop"] == crop]
    f_scores = anova_f_by_band(crop_data, "Stage")
    stage_by_crop_rows.extend(
        {
            "Crop": crop,
            "wavelength_nm": wavelength,
            "band": band,
            "F_statistic": f_score,
        }
        for wavelength, band, f_score in zip(wavelengths_nm, bands, f_scores)
    )
    fig_stage_by_crop.add_trace(
        go.Scatter(
            x=add_wavelength_gaps(wavelengths_nm),
            y=add_wavelength_gaps(f_scores),
            mode="lines",
            name=crop,
            line={"color": crop_colors[crop]},
            hovertemplate=(
                "Crop=%{fullData.name}<br>Wellenlänge=%{x} nm"
                "<br>ANOVA F für Stage=%{y:.2f}<extra></extra>"
            ),
        )
    )
df_band_anova_stage_by_crop = pd.DataFrame(stage_by_crop_rows)
fig_stage_by_crop.update_layout(
    title="ANOVA-F-Statistik je Wellenlänge für Stage innerhalb jedes Crop",
    xaxis_title="Wellenlänge [nm]",
    yaxis_title="ANOVA-F-Statistik (höher bedeutet stärkere Unterschiede zwischen Stadien)",
    width=1000,
    height=550,
)
fig_stage_by_crop.update_xaxes(tickmode="array", tickvals=wavelengths_nm[::10])
fig_stage_by_crop.update_yaxes(type="log")
fig_stage_by_crop.show()

crop_by_stage_rows = []
fig_crop_by_stage = go.Figure()
for stage in stages:
    stage_data = df_train.loc[df_train["Stage"] == stage]
    f_scores = anova_f_by_band(stage_data, "Crop")
    crop_by_stage_rows.extend(
        {
            "Stage": stage,
            "wavelength_nm": wavelength,
            "band": band,
            "F_statistic": f_score,
        }
        for wavelength, band, f_score in zip(wavelengths_nm, bands, f_scores)
    )
    fig_crop_by_stage.add_trace(
        go.Scatter(
            x=add_wavelength_gaps(wavelengths_nm),
            y=add_wavelength_gaps(f_scores),
            mode="lines",
            name=stage,
            line={"color": stage_colors[stage]},
            hovertemplate=(
                "Stage=%{fullData.name}<br>Wellenlänge=%{x} nm"
                "<br>ANOVA F für Crop=%{y:.2f}<extra></extra>"
            ),
        )
    )
df_band_anova_crop_by_stage = pd.DataFrame(crop_by_stage_rows)
fig_crop_by_stage.update_layout(
    title="ANOVA-F-Statistik je Wellenlänge für Crop innerhalb jeder Stage",
    xaxis_title="Wellenlänge [nm]",
    yaxis_title="ANOVA-F-Statistik (höher bedeutet stärkere Unterschiede zwischen Kulturen)",
    width=1000,
    height=550,
)
fig_crop_by_stage.update_xaxes(tickmode="array", tickvals=wavelengths_nm[::10])
fig_crop_by_stage.update_yaxes(type="log")
fig_crop_by_stage.show()

 ## 17.3 Bereiche mit größten Unterschieden
Bereich 600-700nm (beide); 742-923nm , 983 - 1094; 1518 - 1760, 2063 - 2345 (besonders für Stage) wichtig zur unterscheidung von targets (Stage/Crop) <br>
712/722nm sind nahezu bedeutungslos, da sich hier alle 

# 19 Ausreißer und Auffälligkeiten



## 19.1 Ausreißer je Klasse markiert (z. B. Isolation Forest), Spitzen/Sprünge, verrauschte Bänder
### 19.1.1 Isolation Forest / Maha for anomaly detection
Empfehlung für Bereinigung: Mahalanobis Distanz oder isolation forest nutzen (evt. automatisch in der pipeline) <br>
Beide kommen zu unterschidlichen Ergebnissen!!!

In [ ]:

band_columns = sorted(
    [column for column in df_train.columns if column.startswith("X") and column[1:].isdigit()],
    key=lambda column: int(column[1:]),
)

spectra = df_train[band_columns].astype(float).copy()
if not np.isfinite(spectra.to_numpy()).all():
    raise ValueError("Handle missing values before anomaly detection.")

class_key = (
    df_train["Crop"].fillna("NA").astype(str)
    + " / "
    + df_train["Stage"].fillna("NA").astype(str)
)

iso = IsolationForest(
    n_estimators=300,
    contamination="auto",
    max_samples="auto",
    random_state=42,
)
iso.fit(spectra)

anomaly_score = -iso.score_samples(spectra)
df_train = df_train.copy()
df_train["anomaly_score"] = anomaly_score
df_train["class_key"] = class_key
df_train["is_outlier"] = iso.predict(spectra) == -1

fig = go.Figure()
for index, (label, group) in enumerate(df_train.groupby("class_key", sort=True)):
    color = px.colors.qualitative.Plotly[index % len(px.colors.qualitative.Plotly)]
    class_size = len(group)
    customdata = [
        (row_id, float(score), class_size)
        for row_id, score in zip(group["id"].astype(str), group["anomaly_score"])
    ]
    fig.add_trace(
        go.Box(
            y=group["anomaly_score"],
            name=label,
            boxpoints="outliers",
            jitter=0.25,
            marker_color=color,
            customdata=customdata,
            hovertemplate=(
                "<b>%{x}</b><br>ID=%{customdata[0]}"
                "<br>n=%{customdata[2]}"
                "<br>Anomaliewert=%{customdata[1]:.4f}<extra></extra>"
            ),
        )
    )

fig.update_layout(
    title="Isolation-Forest-Anomaliewert nach Crop × Stage",
    xaxis_title="Crop / Stage",
    yaxis_title="Anomaliewert",
    template="plotly_white",
    height=500,
    width=1000,
)
fig.show()

outliers = (
    df_train.loc[df_train["is_outlier"], ["Crop", "Stage", "id", "anomaly_score"]]
    .sort_values("anomaly_score", ascending=False)
    .head(12)
    .reset_index(drop=True)
)
outliers


In [ ]:
mahalanobis_by_class = pd.Series(np.nan, index=df_train.index, dtype=float)
class_outlier_rows = []

band_columns = sorted(
    [column for column in df_train.columns if column.startswith("X") and column[1:].isdigit()],
    key=lambda column: int(column[1:]),
)

spectra = df_train[band_columns].astype(float).copy()
if not np.isfinite(spectra.to_numpy()).all():
    raise ValueError("Handle missing values before Mahalanobis distance checks.")

for (crop, stage), group in df_train.groupby(["Crop", "Stage"], sort=True, observed=True):
    group_index = group.index
    data = group[band_columns].to_numpy(dtype=float)
    if len(data) < 2:
        continue

    class_mean = data.mean(axis=0)
    covariance = np.cov(data, rowvar=False)
    covariance = covariance + np.eye(covariance.shape[0]) * 1e-6
    inv_covariance = np.linalg.pinv(covariance)
    centered = data - class_mean
    distances = np.sqrt(np.einsum("ij,jk,ik->i", centered, inv_covariance, centered))
    mahalanobis_by_class.loc[group_index] = distances

    threshold = np.quantile(distances, 0.99)
    for sample_id, distance in zip(group["id"], distances):
        if distance >= threshold:
            class_outlier_rows.append(
                {
                    "Crop": crop,
                    "Stage": stage,
                    "id": sample_id,
                    "mahalanobis_distance": float(distance),
                }
            )

mahalanobis_outliers = pd.DataFrame(class_outlier_rows).sort_values(
    "mahalanobis_distance", ascending=False
).reset_index(drop=True)

df_train = df_train.copy()
df_train["mahalanobis_distance"] = mahalanobis_by_class

class_key = (
    df_train["Crop"].fillna("NA").astype(str)
    + " / "
    + df_train["Stage"].fillna("NA").astype(str)
)
df_train["class_key"] = class_key

fig = go.Figure()
for label, group in df_train.groupby("class_key", sort=True):
    color = px.colors.qualitative.Plotly[len(fig.data) % len(px.colors.qualitative.Plotly)]
    class_size = len(group)
    customdata = [
        (row_id, float(distance), class_size)
        for row_id, distance in zip(group["id"].astype(str), group["mahalanobis_distance"])
    ]
    fig.add_trace(
        go.Box(
            y=group["mahalanobis_distance"],
            name=label,
            boxpoints="outliers",
            jitter=0.25,
            marker_color=color,
            customdata=customdata,
            hovertemplate=(
                "<b>%{x}</b><br>ID=%{customdata[0]}"
                "<br>Crop/Stage=%{fullData.name}"
                "<br>n=%{customdata[2]}"
                "<br>Mahalanobis-Abstand=%{customdata[1]:.4f}<extra></extra>"
            ),
        )
    )

fig.update_layout(
    title="Mahalanobis-Abstand zum Zentrum der Crop-×-Stage-Gruppe",
    xaxis_title="Crop / Stage",
    yaxis_title="Mahalanobis-Abstand",
    template="plotly_white",
    height=550,
    width=1100,
)
fig.show()

mahalanobis_outliers.head(15)



### 19.2 Sprung an der Überlappung X912–X925
-> docs <br>
Womöglich technischer Grund

## 19.3 Leakage-Risiko bei fast identischen Spektren innerhalb einer Gruppe
Bei einer Cosinus-Ähnlichkeit von mindestens 0,9995 wurden 37.145 Spektrenpaare innerhalb derselben Crop-×-Stage-Gruppe gefunden. Besonders viele Paare gibt es bei Mais – Late (14.738), Sojabohne – Critical (8.815), Mais – Critical (5.369) und Baumwolle – Early_Mid (1.881). Die absoluten Zahlen hängen auch von der Gruppengröße ab: Bezogen auf alle möglichen Paare fällt Reis – Late besonders auf; dort überschreiten rund 21 % der Paare den Schwellenwert.

Das weist auf starke spektrale Ähnlichkeit beziehungsweise mögliche Redundanz innerhalb dieser Gruppen hin. Die Paarzahlen entsprechen jedoch nicht der Zahl einzigartiger betroffener Proben: Eine Probe kann in vielen Paaren vorkommen. Außerdem vergleicht diese Analyse nur Proben innerhalb derselben Crop-×-Stage-Gruppe im Trainingsdatensatz. Sie prüft daher weder Verwechslungen zwischen verschiedenen Klassen noch direkte Ähnlichkeiten zwischen Train und Test.

Ein Leakage-Risiko entsteht, falls solche ähnlichen Proben abhängige Beobachtungen sind – zum Beispiel wiederholte Messungen derselben Fläche – und bei der späteren Aufteilung auf Training und Validierung beziehungsweise Test verteilt werden. Das Modell könnte dann sehr ähnliche Spektren bereits im Training gesehen haben, wodurch die gemessene Leistung zu optimistisch ausfällt. Die vorliegende Analyse belegt das jedoch nicht: Sie wurde nur innerhalb des Trainingsdatensatzes berechnet, und ohne Feld- oder Aufnahme-IDs lässt sich nicht feststellen, ob ähnliche Spektren tatsächlich zusammengehören. Fast identische Spektren sollten daher nicht automatisch gelöscht werden; falls sich gemeinsame Beobachtungsgruppen anhand von Metadaten bestimmen lassen, sollten diese bei der Aufteilung zusammengehalten werden.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

band_columns = sorted(
    [column for column in df_train.columns if column.startswith("X") and column[1:].isdigit()],
    key=lambda column: int(column[1:]),
)

spectra = df_train[band_columns].astype(float).copy()
if not np.isfinite(spectra.to_numpy()).all():
    raise ValueError("Handle missing values before similarity checks.")

similarity_rows = []
for (crop, stage), group in df_train.groupby(["Crop", "Stage"], sort=True, observed=True):
    if len(group) < 2:
        continue
    group_spectra = group[band_columns].astype(float).to_numpy()
    similarity = cosine_similarity(group_spectra)
    upper = np.triu_indices(len(group), k=1)
    scores = similarity[upper]
    pair_positions = np.where(scores >= 0.9995)[0]
    for pos in pair_positions:
        i, j = upper[0][pos], upper[1][pos]
        sample_i = group.iloc[i].to_dict()
        sample_j = group.iloc[j].to_dict()
        similarity_rows.append(
            {
                "Crop": crop,
                "Stage": stage,
                "sample_i": sample_i.get("id"),
                "sample_j": sample_j.get("id"),
                "similarity": float(scores[pos]),
            }
        )

near_duplicate_pairs = pd.DataFrame(similarity_rows)
if near_duplicate_pairs.empty:
    print("Keine fast identischen Spektren oberhalb des Cosinus-Schwellenwerts gefunden.")
else:
    print(f"Gefunden: {len(near_duplicate_pairs)} Spektrenpaare (Cosinus-Ähnlichkeit >= 0,9995).")
    display(near_duplicate_pairs.head(20))

pair_counts = (
    near_duplicate_pairs.groupby(["Crop", "Stage"], dropna=False).size()
    .reset_index(name="near_duplicate_pairs")
    if not near_duplicate_pairs.empty
    else pd.DataFrame(columns=["Crop", "Stage", "near_duplicate_pairs"])
)

fig = px.bar(
    pair_counts,
    x="Crop",
    y="near_duplicate_pairs",
    color="Stage",
    barmode="group",
    title="Fast identische Spektren nach Crop × Stage (Cosinus-Ähnlichkeit >= 0,9995)",
    labels={"near_duplicate_pairs": "Spektrenpaare"},
)
fig.show()


# Zusatz

In [ ]:
from math import ceil

import plotly.graph_objects as go
from plotly.subplots import make_subplots

BANDS = [
    column for column in df_train.columns
    if column.startswith("X") and column[1:].isdigit()
]
WAVELENGTHS = [int(column[1:]) for column in BANDS]
GROUPS = list(df_train.groupby(["Crop", "Stage"], sort=True, observed=True))

N_COLS = 2  # Set to 1, 2, or 3.
if N_COLS not in (1, 2, 3):
    raise ValueError("N_COLS must be 1, 2, or 3")


def make_spectral_figure(title):
    """Create a subplot grid with one panel per observed crop-stage pair."""
    n_cols = min(N_COLS, len(GROUPS))
    n_rows = ceil(len(GROUPS) / n_cols)
    subplot_titles = [
        f"{crop} — {stage} (n={len(group)})"
        for (crop, stage), group in GROUPS
    ]
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=subplot_titles,
        shared_xaxes=True,
        shared_yaxes=True,
    )
    fig.update_layout(
        title=title,
        height=max(350, 300 * n_rows),
        width=500 * n_cols,
        showlegend=False,
        hovermode="closest",
    )
    return fig, n_cols

In [ ]:
fig, n_cols = make_spectral_figure(
    "Reflektanzverteilung nach Wellenlänge, Crop und Stage"
)

ordered_bands = sorted(BANDS, key=lambda band: int(band[1:]))
wavelengths_nm = [int(band[1:]) for band in ordered_bands]
box_width = 0.8 * min(
    right - left for left, right in zip(wavelengths_nm, wavelengths_nm[1:])
)

for group_number, (_, group) in enumerate(GROUPS):
    fig.add_trace(
        go.Box(
            x=wavelengths_nm * len(group),
            y=group[ordered_bands].to_numpy().ravel(),
            width=box_width,
            boxpoints=False,
            hovertemplate="Wellenlänge=%{x} nm<br>Reflektanz=%{y}<extra></extra>",
        ),
        row=group_number // n_cols + 1,
        col=group_number % n_cols + 1,
    )

fig.update_xaxes(
    type="linear",
    title_text="Wellenlänge [nm]",
    tickmode="array",
    tickvals=wavelengths_nm[::10],
)
fig.update_yaxes(title_text="Reflektanz [%]")
fig.show()

## EDA Fazit: Synthese & Cleaning-Empfehlung

### Zusammenfassung der Befunde

Die Trainingsdaten enthalten 5.591 gelabelte Beobachtungen und der Testdatensatz 1.397 Beobachtungen. Von 198 Spektralbändern sind in beiden Datensätzen 67 vollständig leer; 131 Bänder enthalten Messwerte. In sechs weiteren verfügbaren Bändern fehlen einzelne Werte; insgesamt sind 44 Trainings- und 11 Testzeilen betroffen. Im Training wurden zwei Paare identischer Eingaben mit übereinstimmenden Labels gefunden. Zwischen vollständigen Train- und Test-Spektren der verfügbaren Bänder gab es keine exakten Treffer.

Die Zielklassen sind unausgewogen: `rice` umfasst 93 Beobachtungen, `Harvest` 180. Das kleinste beobachtete Crop-×-Stage-Paar (`cotton` × `Harvest`) umfasst 11 Beobachtungen; 23 der 30 möglichen Kombinationen kommen im Training vor. Die mittleren Spektren unterscheiden sich zwischen Gruppen, überlappen jedoch sichtbar. Benachbarte verfügbare Bänder sind stark korreliert; die ersten zwei standardisierten Hauptkomponenten erklären über 90 % der spektralen Varianz, trennen die Klassen in der Projektion aber nicht vollständig. `AEZ` und `Month` hängen ebenfalls mit der Labelverteilung zusammen. Diese explorativen Muster begründen Modellvarianten, belegen aber keine Modellgüte.

Im Similarity-Screen wurden innerhalb derselben Crop-×-Stage-Gruppe 37.145 Paare mit Cosinus-Ähnlichkeit mindestens 0,9995 gefunden. Besonders viele Paare traten bei `corn` × `Late` (14.738), `soybean` × `Critical` (8.815) und `corn` × `Critical` (5.369) auf. Bezogen auf alle möglichen Paare fiel `rice` × `Late` mit rund 21 % auf. Die Schwelle ist explorativ, Cosinus-Ähnlichkeit misst vor allem die Spektrenform und die Paarzahlen sind nicht die Anzahl einzigartiger Proben. Das Ergebnis belegt weder Duplikate noch gemeinsame Felder oder Leakage. Die auffälligen Stellen bei `X912`, `X915` und `X923` sind ebenfalls Prüfkandidaten; ein Sensorartefakt oder eine biologische Ursache ist mit den vorliegenden Angaben nicht belegt.

### Cleaning- und Validierungsempfehlungen

| Thema | Empfehlung | Begründung |
| --- | --- | --- |
| Vollständig leere Bänder | Die 67 vollständig leeren Bänder aus den Modellmerkmalen entfernen; die 131 verfügbaren zunächst als Rohband-Baseline behalten. | Leere Spalten enthalten keine Messinformation. 
| Fehlende Einzelwerte | Die verbleibenden Lücken innerhalb eines Spektrums interpolieren und Randlücken mit dem nächstgelegenen vorhandenen Wert behandeln. Die Imputation innerhalb der Modellpipeline durchführen. | Es fehlen einzelne Bandwerte in wenigen Zeilen. Vorverarbeitung muss innerhalb jedes Trainingsfolds angepasst werden, damit keine Validierungsinformation einfließt.|
| Glättung | Keine Glättung als Standard-Cleaning einführen. Falls begründet, wenige vorab festgelegte Varianten als eigene Modellvarianten in der Cross-Validation vergleichen. (Meilenstein 2) | Lokale Sprung-/Unruhekandidaten belegen keinen allgemeinen Glättungsbedarf. Glättung könnte auch informative spektrale Unterschiede abschwächen. |
| Ausreißer | Isolation-Forest- und klassenweise Mahalanobis-Scores zum Markieren und Prüfen verwenden; Zeilen nicht allein aufgrund eines Scores automatisch löschen. | Ein hoher Score beweist keinen Messfehler. Besonders Mahalanobis-Abstände sind bei vielen Bändern und kleinen Crop-×-Stage-Gruppen vorsichtig zu interpretieren. |
| Exakte Duplikate | Die zwei identischen Trainingspaare vor dem Split deduplizieren, sofern die Labels übereinstimmen. Near-Duplicates nicht anhand eines frei gewählten Schwellenwerts löschen. | Exakte Wiederholungen könnten sonst auf verschiedene Split-Teile geraten. Spektralähnlichkeit allein beweist weder fehlerhafte Daten noch dieselbe Beobachtungsquelle. |
| Ähnliche Spektren / Leakage | Ähnlichkeit als Prüfhypothese dokumentieren. Falls Feld-, Bild- oder Aufnahme-IDs verfügbar sind, abhängige Beobachtungen gemeinsam splitten. | Innerhalb einer Klasse zeigen ähnliche Spektren zunächst mögliche Redundanz. Eine Überschneidung abhängiger Messungen über Split-Grenzen könnte die Evaluation optimistisch machen; die vorhandenen Daten belegen eine solche Abhängigkeit nicht. |
| Split und Evaluation | Den kombinierten Crop-×-Stage-Label beim stratifizierten Split berücksichtigen und Cross-Validation nur auf dem Trainingsanteil durchführen. Balanced Accuracy und Macro-F1 jeweils für Crop und Stage berichten. | Die Klassen sind unausgewogen und das kleinste beobachtete Paar umfasst 11 Zeilen. Stratifizierung hilft, seltene Paare zu erhalten, garantiert aber keine räumlich unabhängige Evaluation. |
| `AEZ` und `Month` | Varianten mit und ohne Kontextmerkmale vergleichen. | Die Merkmale hängen mit der Labelverteilung zusammen; ihre Verfügbarkeit und Stabilität beim späteren Einsatz sind gesondert zu klären. |

### Data Understanding – Textbaustein für den M1-Kurzbericht

> Die Trainingsdaten umfassen 5.591 gelabelte Beobachtungen mit 198 geordneten Spektralbändern von 427 bis 2.395 nm sowie den Kontextmerkmalen `AEZ` und `Month`; der Testdatensatz umfasst 1.397 Beobachtungen. In beiden Datensätzen sind dieselben 67 Bänder vollständig leer. In sechs weiteren verfügbaren Bändern fehlen einzelne Werte; davon sind 44 Trainings- und 11 Testzeilen betroffen. Im Training wurden zwei Paare identischer Eingaben mit übereinstimmenden Labels gefunden. Zwischen vollständigen Spektren der verfügbaren Bänder in Train und Test gab es keine exakten Treffer. Die Zielverteilung ist unausgewogen: `rice` umfasst 93 Beobachtungen, `Harvest` 180, und das kleinste beobachtete Crop-×-Stage-Paar umfasst 11 Beobachtungen; sieben der 30 möglichen Paare sind im Training nicht vertreten. Die mittleren Spektren unterscheiden sich zwischen Gruppen, überlappen aber sichtbar. `AEZ` und `Month` stehen ebenfalls mit der Labelverteilung in Zusammenhang. Für die Modellierung werden fehlende Einzelwerte innerhalb der Pipeline behandelt, exakte Duplikate vor dem Split geprüft und seltene beobachtete Paare beim stratifizierten Split berücksichtigt. Ähnlichkeit zwischen Spektren ist ein Prüfhinweis, aber ohne Feld- oder Aufnahme-IDs kein Nachweis für gemeinsame Beobachtungsquellen oder Leakage.
